# NeuroSLM — Bio-Inspired Language Model (~258M)

Neuroscience-inspired LM with genome-encoded algorithms, differential attention, mixture-of-depths, NT-modulated attention, predictive coding, Hebbian traces, and epigenetic self-optimization.

**Training resumes automatically across runtime disconnects.**
Checkpoints + memory + evolved DNA are persisted to Git LFS.

| Preset | Params | GPU | VRAM | Batch |
|--------|--------|-----|------|-------|
| `large` | ~100M | T4 | 15GB | 4 |
| **`xl`** | **~258M** | **A100** | **40GB** | **4** |

**Novel mechanisms (no existing model has all of these):**
1. Differential attention (dual-softmax noise cancellation)
2. Mixture-of-Depths (dynamic per-token layer skipping)
3. NT-modulated attention temperature (per-head, state-dependent)
4. Predictive coding inter-layer loss (deep supervision)
5. Hebbian fast-weight attention traces (in-context learning accelerator)
6. Genome-encoded module algorithms (evolvable via epigenetic optimizer)

**Setup:** Runtime → Change runtime type → **A100** → Set `GITHUB_TOKEN` in cell 2

In [ ]:
# 1) Accelerator + torch check
import subprocess, sys

print(f'Python {sys.version}')

# Check CUDA
r = subprocess.run(
    [sys.executable, "-c",
     "import torch; print(torch.__version__); print(torch.cuda.is_available())"],
    capture_output=True, text=True
)
lines = r.stdout.strip().split('\n')
torch_ver = lines[0] if lines else '?'
has_cuda  = len(lines) > 1 and lines[1] == 'True'
print(f'torch {torch_ver}')

# Check TPU/XLA if no CUDA
has_tpu = False
if not has_cuda:
    r2 = subprocess.run(
        [sys.executable, "-c",
         "import torch_xla.core.xla_model as xm; print(xm.xla_device())"],
        capture_output=True, text=True
    )
    has_tpu = r2.returncode == 0 and bool(r2.stdout.strip())
    if has_tpu:
        print(f'TPU device: {r2.stdout.strip()}')

if has_cuda:
    get_ipython().system('nvidia-smi')
    import torch
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    DEVICE = "cuda"
    print(f'\n✓ GPU: {name} ({vram:.1f} GB)')
    if vram < 35:
        print('⚠️  <40 GB VRAM — use preset="large" (100M) instead of "xl"')
    else:
        print('✓ Ready for xl model')
elif has_tpu:
    DEVICE = "xla"
    print(f'\n✓ TPU detected — DEVICE={DEVICE!r}')
    print('  Tip: v2/v3 TPUs prefer preset="large"; v4/v5 handle "xl"')
else:
    DEVICE = "cpu"
    print('\n' + '='*60)
    print('❌ No GPU or TPU found!')
    print('FIX: Runtime → Disconnect and delete runtime')
    print('     Runtime → Change runtime type → A100 / T4 / TPU')
    print('     Then re-run all cells from the top.')
    print('='*60)

print(f'\nDEVICE = {DEVICE!r}')


In [ ]:
# 2) Install deps + clone brian repo + Git LFS + restore checkpoints + DNA
import torch, os, glob, shutil, subprocess, sys

# Accept CUDA or TPU; only block on plain CPU
_has_cuda = torch.cuda.is_available()
_has_tpu  = False
if not _has_cuda:
    _r_t = subprocess.run([sys.executable, "-c",
         "import torch_xla.core.xla_model as xm; xm.xla_device(); print('ok')"],
        capture_output=True, text=True)
    _has_tpu = _r_t.returncode == 0 and "ok" in _r_t.stdout

if not _has_cuda and not _has_tpu:
    raise RuntimeError(
        "No GPU or TPU! Go to Runtime -> Disconnect and delete runtime -> "
        "Change runtime type -> A100/T4 or TPU -> re-run from cell 1"
    )

if _has_cuda:
    print("torch " + torch.__version__ + "  CUDA GPU=" + torch.cuda.get_device_name(0))
else:
    print("torch " + torch.__version__ + "  TPU/XLA backend")

# Install deps (torch is already from Colab — don't touch it)
get_ipython().system("pip install -q numpy tiktoken datasets tqdm einops networkx transformers")
get_ipython().system("pip install -q --no-deps accelerate")
if _has_tpu:
    get_ipython().system("pip install -q cloud-tpu-client")

# ─── Clone the brian repo (new name) ──────────────────────────────────────
REPO_URL  = "https://github.com/269652/brian.git"
REPO_DIR  = "/content/brian"

# Auth from Colab secret if available (allows private repo + push later)
_token = ""
try:
    from google.colab import userdata as _ud
    _token = (_ud.get("GITHUB") or "").strip()
except Exception:
    _token = os.environ.get("GITHUB", "").strip()

_auth_url = REPO_URL.replace("https://", "https://" + _token + "@") if _token else REPO_URL

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Cloning " + REPO_URL + " -> " + REPO_DIR)
    _clone = subprocess.run(["git", "clone", _auth_url, REPO_DIR],
                            capture_output=True, text=True)
    if _clone.returncode != 0:
        raise RuntimeError("git clone failed: " + _clone.stderr)
    print("Repo cloned.")
else:
    print("Repo already present at " + REPO_DIR + ", pulling latest")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
if _token:
    os.environ["GITHUB"] = _token
    os.environ["GITHUB_TOKEN"] = _token

# Git LFS
get_ipython().system("apt-get install -y git-lfs -qq 2>/dev/null")
get_ipython().system("git lfs install")
get_ipython().system("git lfs pull")

# ─── Checkpoint dir: directly inside the repo under lfs_checkpoints ──────
CKPT_DIR = "/content/brian/lfs_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

# Restore LFS checkpoints + memory + evolved DNA (already in place after lfs pull)
existing = sorted(glob.glob(CKPT_DIR + "/*.pt"))
dna_files = sorted(glob.glob(CKPT_DIR + "/*.dna.json"))
if existing:
    print(str(len(existing)) + " checkpoint(s). Latest: " + existing[-1])
    ckpt = torch.load(existing[-1], map_location="cpu", weights_only=False)
    print("  Step " + str(ckpt.get("step", "?")))
    has_dna = "module_genomes" in ckpt or "epigenetic" in ckpt
    print("  Evolved DNA in checkpoint: " + str(has_dna))
    if dna_files:
        print("  DNA snapshots: " + str(len(dna_files)) + " (.dna.json files)")
    del ckpt
else:
    print("No checkpoints in " + CKPT_DIR + " — training from scratch")

In [ ]:
# 2b) Pull latest code + notebook from GitHub (keeps LFS files + runtime)
# Re-run this whenever you've pushed changes from your machine. It updates
# the .py code and the colab_run.ipynb FILE on disk WITHOUT re-cloning,
# WITHOUT re-downloading the already-pulled LFS checkpoints, and WITHOUT
# restarting the Python runtime.
#
# IMPORTANT: pulling updates the notebook file on disk, but Colab won't
# hot-swap the cells you're running. To load updated cells:
#   File > Reload from disk   (Jupyter)  /  re-open the notebook URL (Colab).
# Updated .py modules ARE picked up by the next `python -m neuroslm.*` cell.
import os, subprocess
REPO_DIR = "/content/brian"
os.chdir(REPO_DIR)

# Don't re-smudge (re-download) LFS blobs on checkout - we keep what we have.
os.environ["GIT_LFS_SKIP_SMUDGE"] = "1"

# Auth: prefer the Colab secret / env token so private pulls work.
_tok = os.environ.get("GITHUB", "").strip()
if not _tok:
    try:
        from google.colab import userdata as _ud
        _tok = (_ud.get("GITHUB") or "").strip()
    except Exception:
        _tok = ""
if _tok:
    os.environ["GITHUB"] = _tok
    subprocess.run(["git", "remote", "set-url", "origin",
                    "https://x-access-token:" + _tok + "@github.com/269652/brian.git"],
                   check=False)

# Rebase onto origin/master. --autostash shelves any local working-tree
# changes (e.g. checkpoints training just wrote) and restores them after,
# so a fast-moving lfs_checkpoints dir can't block the pull.
_pull = subprocess.run(
    ["git", "-c", "credential.helper=", "pull", "--rebase", "--autostash",
     "origin", "master"],
    capture_output=True, text=True)
out = (_pull.stdout or "") + (_pull.stderr or "")
# Redact token if it ever leaks into output.
if _tok:
    out = out.replace(_tok, "***")
print(out.strip())

print("\n--- latest commits ---")
_log = subprocess.run(["git", "log", "--oneline", "-6"],
                      capture_output=True, text=True)
print(_log.stdout.strip())
print("\nReload the notebook (File > Reload from disk) to pick up updated cells.")


In [ ]:
# 3) Verify repo / checkpoint state (safe to re-run)
import os, glob, torch

REPO_DIR = "/content/brian"
CKPT_DIR = "/content/brian/lfs_checkpoints"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    raise RuntimeError("Repo not cloned yet — run cell 2 first.")

os.chdir(REPO_DIR)
print("In repo dir: " + os.getcwd())

ckpts = sorted(glob.glob(CKPT_DIR + "/*.pt"))
mems  = sorted(glob.glob(CKPT_DIR + "/*.mem"))
dnas  = sorted(glob.glob(CKPT_DIR + "/*.dna.json"))
print("  .pt        files: " + str(len(ckpts)))
print("  .mem       files: " + str(len(mems)))
print("  .dna.json  files: " + str(len(dnas)))
if ckpts:
    latest = ckpts[-1]
    print("Latest checkpoint: " + latest)
    ckpt = torch.load(latest, map_location="cpu", weights_only=False)
    print("  Step: " + str(ckpt.get("step", "?")))
    print("  Preset: " + str(ckpt.get("preset", "?")))
    del ckpt

In [ ]:
# 4) Configuration — change these to control training
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

PRESET     = "xl"        # "large" (100M, T4/TPU v2-3) | "xl" (350M, A100/TPU v4+)
TOTAL_STEPS = 2000
BATCH_SIZE  = 1
GRAD_ACCUM  = 16         # effective batch = BATCH_SIZE * GRAD_ACCUM
SAVE_EVERY  = 500
CKPT_DIR    = "/content/brian/lfs_checkpoints"   # write directly into the repo
MODE        = "mix"
CHAT_RATIO  = 0.6
OVERWRITE_CKPT = False   # True = overwrite existing checkpoint; False = step-numbered files

import torch, os
os.makedirs(CKPT_DIR, exist_ok=True)

# DEVICE was set in cell 1; fall back gracefully if cells run out of order
if "DEVICE" not in dir():
    if torch.cuda.is_available():
        DEVICE = "cuda"
    else:
        import subprocess as _sp2, sys as _sy2
        _r2 = _sp2.run(
            [_sy2.executable, "-c",
             "import torch_xla.core.xla_model as xm; xm.xla_device(); print('ok')"],
            capture_output=True, text=True)
        DEVICE = "xla" if (_r2.returncode == 0 and "ok" in _r2.stdout) else "cpu"

if DEVICE == "cuda":
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram < 35 and PRESET == "xl":
        print("Only " + str(int(vram)) + " GB VRAM — falling back to large preset (100M)")
        PRESET     = "large"
        BATCH_SIZE = 1
    else:
        print(str(int(vram)) + " GB VRAM — using " + PRESET + " preset (batch=" + str(BATCH_SIZE) + ")")
elif DEVICE == "xla":
    print("TPU runtime — using " + PRESET + " preset (batch=" + str(BATCH_SIZE) + ")")
    print("  Tip: v2/v3 HBM is 8 GB/core; set PRESET=\"large\" if OOM")

print("")
print("Training plan:")
print("  Preset:     " + PRESET)
print("  Device:     " + DEVICE)
print("  Steps:      " + str(TOTAL_STEPS))
print("  Batch:      " + str(BATCH_SIZE) + " x " + str(GRAD_ACCUM) + " grad_accum = " + str(BATCH_SIZE*GRAD_ACCUM) + " effective")
print("  Mode:       " + MODE + " (chat_ratio=" + str(CHAT_RATIO) + ")")
print("  Save every: " + str(SAVE_EVERY))
print("  CKPT dir:   " + CKPT_DIR)
print("  Overwrite:  " + str(OVERWRITE_CKPT))

In [ ]:
# 5) ABLATION: Baseline vs Full Model (1000 steps each)
# ╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋╋
# Trains both models for 1000 steps to confirm bio modules improve over a
# param-matched vanilla transformer BEFORE committing to the full 100K run.
#
# Checkpoint policy:
#   - FULL MODEL writes to CKPT_DIR (cell 6 resumes from here).
#     - save+push every 200 steps, log every 10 steps (fine-grained diagnostics
#       while we tune the bio-pipeline mechanics).
#   - BASELINE writes to a separate dir for comparison only.
#     - save every 500, log every 50 — it's a sanity check; we don't need
#       per-step detail for a vanilla transformer.
#   - --resume latest in both: if Colab disconnects, re-running the cell
#     continues from the most recent step instead of starting over.
#   - Filename now includes the optimizer tag, so AdamW and Adafactor
#     streams coexist without ever clashing.
import os, subprocess

# Refresh credentials from Colab secret (safe to re-run; no-op if already set)
_tok = ""
try:
    from google.colab import userdata as _ud
    _tok = (_ud.get("GITHUB") or "").strip()
except Exception:
    _tok = os.environ.get("GITHUB", "").strip()
if _tok:
    os.environ["GITHUB"] = _tok
    os.environ["GITHUB_TOKEN"] = _tok
    with open(os.path.expanduser("~/.git-credentials"), "w") as _cf:
        _cf.write("https://" + _tok + "@github.com\n")
    subprocess.run(["git", "config", "--global", "credential.helper", "store"], check=False)
    print("Credentials refreshed (" + str(len(_tok)) + " chars)")
else:
    print("No GITHUB token -- checkpoint push will be skipped")

os.environ["PYTHONUNBUFFERED"] = "1"

print("=" * 60)
print("  ABLATION: Baseline vs Full Model (1000 steps each)")
print("  FULL:     save+push every 200, log every 10")
print("  BASELINE: save+push every 500, log every 50")
print("=" * 60)

# --- Full model: all bio modules. Writes to CKPT_DIR so full training resumes from here.
print(f"\n▶ Training FULL MODEL (bio modules) → {CKPT_DIR}")
_fm_cmd = (f"cd /content/brian && python -u -m neuroslm.train"
           f" --preset {PRESET}"
           f" --steps 1000"
           f" --batch_size {BATCH_SIZE}"
           f" --grad_accum {GRAD_ACCUM}"
           f" --ckpt_dir {CKPT_DIR}"
           f" --device {DEVICE}"
           f" --mode {MODE}"
           f" --chat_ratio {CHAT_RATIO}"
           f" --save_every 200"
           f" --log_every 10"
           f" --seed 42"
           f" --optimizer adamw"
           f" --resume latest"
           + (" --overwrite_ckpt" if OVERWRITE_CKPT else ""))
print(_fm_cmd)
!{_fm_cmd}

# --- Baseline: param-matched vanilla transformer. Separate dir (comparison only).
print("\n▶ Training BASELINE (param-matched vanilla transformer) → /content/checkpoints_baseline")
_bl_cmd = (f"cd /content/brian && python -u -m neuroslm.train"
           f" --preset {PRESET}"
           f" --steps 1000"
           f" --batch_size {BATCH_SIZE}"
           f" --grad_accum {GRAD_ACCUM}"
           f" --ckpt_dir /content/checkpoints_baseline"
           f" --device {DEVICE}"
           f" --mode {MODE}"
           f" --chat_ratio {CHAT_RATIO}"
           f" --save_every 500"
           f" --log_every 50"
           f" --seed 42"
           f" --optimizer adamw"
           f" --resume latest"
           f" --baseline"
           + (" --overwrite_ckpt" if OVERWRITE_CKPT else ""))
print(_bl_cmd)
!{_bl_cmd}

print(f"\n✓ Both ablation runs complete.")
print(f"  Full-model checkpoint in {CKPT_DIR} → cell 6 will resume from here.")
print(f"  Baseline checkpoint in /content/checkpoints_baseline (comparison only).")
print(f"  Run cell 5b to compare.")

In [ ]:
# 5b) Compare ablation results
import glob, torch

print("=" * 60)
print("  ABLATION RESULTS: 1000 steps")
print("=" * 60)

for label, ckpt_dir in [
    ("BASELINE (param-matched vanilla)", "/content/checkpoints_baseline"),
    ("FULL MODEL  (bio modules)",        CKPT_DIR),
]:
    ckpts = sorted(glob.glob(f'{ckpt_dir}/*.pt'))
    if ckpts:
        ckpt = torch.load(ckpts[-1], map_location='cpu', weights_only=False)
        step = ckpt.get('step', '?')
        cfg_d = ckpt.get('cfg', {})
        n_params = sum(v.numel() for v in ckpt['model'].values())
        is_baseline = cfg_d.get('baseline', False)
        print(f"\n  {label}:")
        print(f"    Path:       {ckpts[-1]}")
        print(f"    Params:     {n_params/1e6:.2f}M")
        print(f"    Step:       {step}")
        print(f"    Baseline:   {is_baseline}")
        del ckpt
    else:
        print(f"\n  {label}: No checkpoint found in {ckpt_dir}/")

print("\n" + "=" * 60)
print("  Check step-by-step loss in the training output above.")
print("  FULL MODEL loss < BASELINE → bio modules help → run cell 6 to continue.")
print("  Cell 6 will resume from the full-model checkpoint in CKPT_DIR.")
print("  FULL MODEL loss ≥ BASELINE → tune loss weights first.")
print("=" * 60)

In [ ]:
# 5c) OOD Perplexity Test - WikiText-103 vs in-distribution PPL
# Prefers the protected `*_best.pt` (lowest-loss checkpoint, never pruned);
# falls back to a fixed step if no best file exists yet.
# Verdict by gap ratio:  <1.5 excellent | <2.0 good | <3.0 possible overfit | else strong overfit.
# Designed for Colab T4 (15 GB) - sliding window, bs=4, @torch.inference_mode.
FALLBACK_STEP = 6000   # used only when no *_best.pt is found

import os, glob
get_ipython().system("pip install -q datasets tqdm")

# Prefer the lowest-loss best checkpoint; else test the fixed fallback step.
_best = sorted(glob.glob(CKPT_DIR + "/*_best.pt"))
# don't count baseline best when testing the full model
_best = [p for p in _best if "_baseline_" not in os.path.basename(p)]
if _best:
    _sel = "--checkpoint " + _best[-1]
    print("Using BEST checkpoint: " + _best[-1])
else:
    _sel = "--step " + str(FALLBACK_STEP)
    print("No *_best.pt found - falling back to step " + str(FALLBACK_STEP))

_cmd = ("cd /content/brian && python -u brian_ood_test.py "
        + _sel + " --batch_size 4 --stride 512 --max_ood_windows 200")
print(_cmd)
get_ipython().system(_cmd)

# Show the saved JSON summary.
import json as _json, os as _os
_results = "/content/brian/ood_test_results.json"
if _os.path.exists(_results):
    with open(_results) as _f:
        _r = _json.load(_f)
    print("\n" + "=" * 60)
    print(" OOD TEST RESULTS (also saved to " + _results + ")")
    print("=" * 60)
    for _k, _v in _r.items():
        print("  " + str(_k) + ": " + str(_v))


In [ ]:
# 5d) OOD eval on the LATEST DSL checkpoint (gap_ratio + ppl)
# Picks the highest-step `dsl_arch_*_step*.pt` from the run directory and
# runs `brian_ood_test.py` against it. The OOD test transparently detects
# DSL vs legacy Brain checkpoints; the WikiText-103 sliding-window pass
# plus an in-distribution train-set sample produce a gap_ratio.
#
# Verdict by gap ratio:
#   <1.5 excellent | <2.0 good | <3.0 possible overfit | >=3 strong overfit
import os, glob, re, json

get_ipython().system("pip install -q datasets tqdm")

CKPT_DIR_DEFAULT = "/content/brian/lfs_checkpoints"
_ckpt_dir = locals().get("CKPT_DIR", CKPT_DIR_DEFAULT)

# Find every `dsl_arch_<run_id>_step<N>.pt` and pick the highest N
_step_re = re.compile(r"_step(\d+)\.pt$")
_dsl = []
for p in glob.glob(os.path.join(_ckpt_dir, "dsl_arch_*_step*.pt")):
    m = _step_re.search(os.path.basename(p))
    if m:
        _dsl.append((int(m.group(1)), p))
_dsl.sort()

if not _dsl:
    print("No dsl_arch_*_step*.pt files in " + _ckpt_dir)
    print("Run training first (deploy via _deploy_train.py or local train_dsl).")
else:
    step, ckpt_path = _dsl[-1]
    print("Latest DSL checkpoint @ step " + str(step) + ": " + ckpt_path)
    print("  (" + str(len(_dsl)) + " total DSL checkpoints in dir)")

    _cmd = ("cd /content/brian && python -u brian_ood_test.py "
            "--checkpoint " + ckpt_path + " "
            "--batch_size 4 --stride 512 --max_ood_windows 200")
    print("\n$ " + _cmd + "\n")
    get_ipython().system(_cmd)

    # Show the JSON summary (also written to ood_test_results.json).
    _results_path = "/content/brian/ood_test_results.json"
    if os.path.exists(_results_path):
        with open(_results_path) as _f:
            _r = json.load(_f)
        print("\n" + "=" * 60)
        print(" OOD TEST RESULTS  (saved to " + _results_path + ")")
        print("=" * 60)
        for _k, _v in _r.items():
            print("  " + str(_k) + ": " + str(_v))
        _gr = _r.get("gap_ratio")
        if isinstance(_gr, (int, float)):
            if _gr < 1.5:
                _verdict = "EXCELLENT (model generalises well)"
            elif _gr < 2.0:
                _verdict = "GOOD"
            elif _gr < 3.0:
                _verdict = "POSSIBLE OVERFIT"
            else:
                _verdict = "STRONG OVERFIT"
            print("\n  Verdict: gap_ratio=" + format(_gr, ".2f") + " -> " + _verdict)


In [ ]:
# 6) Full training — RESUMES AUTOMATICALLY from latest checkpoint
# Run this after the ablation (cells 5 + 5b) confirms bio modules help.
import os, sys, subprocess

# Refresh credentials from Colab secret (safe to re-run; no-op if already set)
_tok = ""
try:
    from google.colab import userdata as _ud
    _tok = (_ud.get("GITHUB") or "").strip()
except Exception:
    _tok = os.environ.get("GITHUB", "").strip()
if _tok:
    os.environ["GITHUB"] = _tok
    os.environ["GITHUB_TOKEN"] = _tok
    with open(os.path.expanduser("~/.git-credentials"), "w") as _cf:
        _cf.write("https://" + _tok + "@github.com\n")
    subprocess.run(["git", "config", "--global", "credential.helper", "store"], check=False)
    print("Credentials refreshed (" + str(len(_tok)) + " chars)")
else:
    print("No GITHUB token -- checkpoint push will be skipped")

os.environ["PYTHONUNBUFFERED"] = "1"

_cmd = ("cd /content/brian && python -u -m neuroslm.train"
        " --preset " + PRESET +
        " --steps " + str(TOTAL_STEPS) +
        " --batch_size " + str(BATCH_SIZE) +
        " --grad_accum " + str(GRAD_ACCUM) +
        " --ckpt_dir " + CKPT_DIR +
        " --device " + DEVICE +
        " --meta"
        " --mode " + MODE +
        " --chat_ratio " + str(CHAT_RATIO) +
        " --save_every " + str(SAVE_EVERY) +
        " --resume latest"
        + (" --overwrite_ckpt" if OVERWRITE_CKPT else ""))
print(_cmd)
get_ipython().system(_cmd)

In [ ]:
# 7) Upload checkpoint + memory + DNA to GitHub
# Reads the GITHUB secret fresh in case env var was lost.
import glob, shutil, os, subprocess

CKPT_DIR = locals().get("CKPT_DIR", "/content/brian/lfs_checkpoints")

# Re-read token (works even if cell 2 was not run this session)
_token = os.environ.get("GITHUB", "").strip()
if not _token:
    try:
        from google.colab import userdata as _ud
        _token = (_ud.get("GITHUB") or "").strip()
        if _token:
            os.environ["GITHUB"] = _token
            os.environ["GITHUB_TOKEN"] = _token
            _creds = "https://" + _token + "@github.com\n"
            with open(os.path.expanduser("~/.git-credentials"), "w") as _f:
                _f.write(_creds)
            subprocess.run(["git", "config", "--global", "credential.helper", "store"], check=False)
    except Exception:
        pass

ckpts = sorted(glob.glob(CKPT_DIR + "/*.pt"))
mems  = sorted(glob.glob(CKPT_DIR + "/*.mem"))
dnas  = sorted(glob.glob(CKPT_DIR + "/*.dna.json"))
files = ckpts[-2:] + mems[-2:] + dnas[-2:]

if not files:
    print("No checkpoints in " + CKPT_DIR + ". Train first (cell 6).")
else:
    if _token:
        remote = "https://" + _token + "@github.com/269652/brian.git"
        subprocess.run(["git", "remote", "set-url", "origin", remote],
                       cwd="/content/brian", check=False)
    else:
        print("Warning: no GITHUB token -- push may fail")

    # Files are already in lfs_checkpoints (CKPT_DIR points there); just add+commit+push
    subprocess.run(["git", "add", "-f", "lfs_checkpoints/"],
                   cwd="/content/brian", check=False)
    r_c = subprocess.run(
        ["git", "commit", "--allow-empty", "-m", "chkpt: " + os.path.basename(files[-1])],
        cwd="/content/brian", capture_output=True, text=True)
    print(r_c.stdout.strip() or r_c.stderr.strip())

    r_p = subprocess.run(["git", "push"], cwd="/content/brian",
                         capture_output=True, text=True)
    if r_p.returncode == 0:
        print("Pushed: " + str([os.path.basename(f) for f in files]))
    else:
        print("Push failed:\n" + r_p.stderr)
        print("If you see denied: your PAT needs repo scope.")

In [ ]:
# 7) Benchmarks — HellaSwag, ARC-Easy, ARC-Challenge, MMLU
# Compares against SmolLM2, TinyLlama, Phi-3, Qwen2.5, Llama-3.1
import glob
ckpts = sorted(glob.glob(CKPT_DIR + '/*.pt'))
if ckpts:
    latest = ckpts[-1]
    print(f"Benchmarking: {latest}")
    !python -m neuroslm.benchmarks --ckpt {latest} --preset {PRESET} --device {DEVICE} --max_samples 500
else:
    print("No checkpoints. Train first.")

In [ ]:
# 8) Generate text
import glob
ckpts = sorted(glob.glob(CKPT_DIR + '/*.pt'))
if ckpts:
    latest = ckpts[-1]
    print(f'Using: {latest}')
    !python -m neuroslm.generate --ckpt {latest} --preset {PRESET} --prompt "Explain the theory of relativity in simple terms" --max_new 512 --temperature 0.7 --top_k 40 --device {DEVICE}
else:
    print('No checkpoints. Train first.')

In [ ]:
# 9) Intelligence Metrics — consciousness, identity drift, causal density
import glob, torch, sys
sys.path.insert(0, '.')
from neuroslm.config import PRESETS
from neuroslm.brain import Brain
from neuroslm.tokenizer import Tokenizer

ckpts = sorted(glob.glob(CKPT_DIR + '/*.pt'))
assert ckpts, "No checkpoints. Train first."
latest = ckpts[-1]

tok = Tokenizer()
cfg = PRESETS[PRESET]()
cfg.vocab_size = tok.vocab_size
brain = Brain(cfg).to(DEVICE)
ckpt = torch.load(latest, map_location="cuda", weights_only=False)
brain.load_state_dict(ckpt["model"], strict=False)
brain.eval()
n_params = sum(p.numel() for p in brain.parameters())
print(f"Model: {n_params/1e6:.1f}M params, step {ckpt.get('step', '?')}")

# Load memory if available
import os
mem_path = latest.replace('.pt', '.mem')
if os.path.exists(mem_path):
    brain.load_memory_checkpoint(mem_path)
    print(f"Memory loaded: {mem_path}")

# Show intelligence metrics
brain.metrics.observe_narrative(brain.narrative_system)
brain.metrics.observe_memory(brain.episodic, brain.consolidated, brain.causal)
snap = brain.metrics.snapshot()
print("\n" + "=" * 60)
print("  INTELLIGENCE & CONSCIOUSNESS METRICS")
print("=" * 60)
for k, v in snap.items():
    if isinstance(v, float):
        print(f"  {k:30s}: {v:.4f}")
    else:
        print(f"  {k:30s}: {v}")
print("=" * 60)
print(f"\n  Causal rules learned: {brain.causal.stats()}")
print(f"  Memory gate stats: {brain.comprehension_gate.stats()}")

In [ ]:
# 10) Inspect Module Algorithms — Genome alleles → Opcodes → Lisp → Module Params
# The genome IS the program. Alleles directly encode opcode logits + operands.
# Pipeline: ModuleGenome.alleles → decompile → Lisp → extract params → push to modules
import glob, torch, sys, os
sys.path.insert(0, '.')
from neuroslm.config import PRESETS
from neuroslm.brain import Brain
from neuroslm.tokenizer import Tokenizer

ckpts = sorted(glob.glob(CKPT_DIR + '/*.pt'))
assert ckpts, "No checkpoints. Train first."
latest = ckpts[-1]

tok = Tokenizer()
cfg = PRESETS[PRESET]()
cfg.vocab_size = tok.vocab_size
brain = Brain(cfg).to("cpu")
ckpt = torch.load(latest, map_location="cpu", weights_only=False)
brain.load_state_dict(ckpt["model"], strict=False)

# Restore module genomes if saved
if 'module_genomes' in ckpt and hasattr(brain, 'module_genomes'):
    from neuroslm.dna.compiler import ModuleGenomePool
    brain.module_genomes = ModuleGenomePool.from_state(ckpt['module_genomes'])
    brain._recompile_all_genomes()
    print(f"✓ Restored module genomes from step {ckpt.get('step', '?')}")

brain.eval()
print(f"Loaded step {ckpt.get('step', '?')}\n")

# Show compilation report
if hasattr(brain, 'genome_compiler'):
    print(brain.genome_compilation_report())

    # Show genome summaries — what each module's algorithm looks like
    print("\n" + "=" * 60)
    print("  MODULE GENOME ALGORITHMS (genome IS the program)")
    print("=" * 60)
    for region, genome in brain.module_genomes.active_all().items():
        print(f"\n  {genome.summary()}")
        print(f"    Generation: {genome.generation}, Fitness: {genome.fitness:.4f}")
        # Show extracted params that actually control the module
        env = brain.genome_compiler.get_env(region)
        param_keys = [k for k in env if not k.startswith('_') and k not in
                      ('__region__', 'layers', 'connections', 'learning_rule',
                       'projections', 'nt_production')]
        if param_keys:
            params_str = ", ".join(f"{k}={env[k]:.3f}" for k in sorted(param_keys)
                                   if isinstance(env.get(k), (int, float)))
            print(f"    Params → module: {params_str}")

    print("\n" + "=" * 60)
    print("  COMPILED LISP (decompiled from genome alleles)")
    print("=" * 60)
    for name, lisp_src in brain.get_all_module_lisp().items():
        print(f"\n{'─' * 60}")
        print(lisp_src)

    # Save to files
    out_dir = brain.genome_compiler.save_all_lisp('compiled_programs')
    print(f"\n✓ Saved compiled .lisp files to {out_dir}/")
    for f in sorted(os.listdir(out_dir)):
        print(f"  {f}")
else:
    print("No genome compiler found (baseline model)")

## Multi-day training workflow

1. **First run:** Run cells 1–6 sequentially. Training starts from scratch.
2. **Runtime disconnects:** Colab disconnects after ~12h. Checkpoints are auto-saved + pushed.
3. **Resume:** Re-run cells 1–5. Cell 5 auto-detects the latest checkpoint and continues.
4. **Track progress:** Run cell 7 (benchmarks) periodically to see improvement vs SOTA.
5. **Ablation:** Run cell 6b instead of 5 to compare baseline vs full model (critical first step).
6. **Target:** 100K steps ≈ 800M tokens at batch=4, ctx=2048.

### Training data pipeline
- **Text:** FineWeb-Edu (10B tokens curated web), Cosmopedia, TinyStories
- **Chat:** OpenHermes-2.5, UltraChat-200k, WildChat-1M, SlimOrca, hh-rlhf, Dolly
- **Mode `mix`:** 60% chat + 40% text (tunable via `CHAT_RATIO`)

### What gets persisted across restarts
- `.pt` checkpoint (model weights + optimizer + genome state + epigenetic marks)
- `.mem` memory checkpoint (episodic + consolidated + causal)
- `.dna.json` evolved DNA snapshot (inspectable JSON)

### Benchmark targets (258M xl preset)
| Benchmark | Random | Target | SOTA ~300M (SmolLM2) |
|-----------|--------|--------|---------------------|
| HellaSwag | 0.25 | >0.35 | 0.42 |
| ARC-Easy | 0.25 | >0.40 | 0.48 |
| ARC-Challenge | 0.25 | >0.28 | 0.30 |
| MMLU | 0.25 | >0.28 | 0.30 |